## High-level GLM

In [6]:
from nilearn.maskers import NiftiMasker
from nilearn.glm.first_level import make_first_level_design_matrix, run_glm
from nilearn.glm.contrasts import compute_contrast
from nilearn.image import math_img
from nilearn import image, plotting
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import datetime
import base64
import glob
import bids
from nilearn.plotting import plot_glass_brain, plot_stat_map
from nilearn.image import new_img_like

layout = bids.BIDSLayout('/Volumes/drive/AVP-BDD/derivatives', validate=False,
                  config=['bids','derivatives'])
print(layout)
subjects = layout.get_subjects()


# -----------------------------
# Configuration
# -----------------------------
runs = ['1', '2', '3']
task = 'SFhigh'  # UPDATE if different

hrf_model = "spm"
high_pass = 0.01

# Quality control thresholds
fd_threshold = 0.5  # mm
accuracy_threshold = 0.80  # 80% minimum accuracy

# Toggles
APPLY_FD_EXCLUSION = True
APPLY_BEHAVIORAL_EXCLUSION = True

# Define conditions for high-level task
CATEGORIES = ['Face', 'Body', 'House']
SPATIAL_FREQUENCIES = ['LSF', 'NSF', 'HSF']

# Define categorical ROIs to load from fLoc
FACE_ROIS = ['FFA', 'OFA', 'FuS']
BODY_ROIS = ['EBA', 'FBA', 'G_S_occ_inf']
HOUSE_ROIS = ['PPA', 'RSC', 'TOS']

# Confound variables
confound_vars = [
    'trans_x','trans_x_derivative1','trans_x_derivative1_power2','trans_x_power2',
    'trans_y','trans_y_derivative1','trans_y_derivative1_power2','trans_y_power2',
    'trans_z','trans_z_derivative1','trans_z_derivative1_power2','trans_z_power2',
    'rot_x','rot_x_derivative1','rot_x_derivative1_power2','rot_x_power2',
    'rot_y','rot_y_derivative1','rot_y_derivative1_power2','rot_y_power2',
    'rot_z','rot_z_derivative1','rot_z_derivative1_power2','rot_z_power2',
    'csf','csf_derivative1','csf_derivative1_power2','csf_power2',
    'white_matter','white_matter_derivative1',
    'white_matter_derivative1_power2','white_matter_power2'
]

# Storage
all_subject_results = {}

# Summaries
fd_summary = {
    'subject': [],
    'total_frames': [],
    'scrubbed_frames': [],
    'pct_scrubbed': []
}

behavioral_summary = {
    'subject': [],
    'run': [],
    'category': [],
    'n_oddballs': [],
    'n_responses': [],
    'accuracy': [],
    'excluded': []
}

glm_summary = {
    'subject': [],
    'group': [],
    'n_v1_voxels': [],
    'n_v1_low_sf_voxels': [],
    'n_v1_high_sf_voxels': [],
    'n_face_roi_voxels': [],
    'n_body_roi_voxels': [],
    'n_house_roi_voxels': []
}

category_run_mapping = {}

def image_to_base64(image_path):
    """Convert image file to base64 string"""
    try:
        with open(image_path, 'rb') as img_file:
            return base64.b64encode(img_file.read()).decode('utf-8')
    except:
        return None

def load_roi_mask(sub, roi_name, roi_dir):
    """Load a categorical ROI mask"""
    roi_path = os.path.join(roi_dir, f'{roi_name}_combined.nii.gz')
    if os.path.exists(roi_path):
        return image.load_img(roi_path)
    return None

def combine_roi_masks(masks):
    """Combine multiple ROI masks into one"""
    valid_masks = [m for m in masks if m is not None]
    if len(valid_masks) == 0:
        return None
    elif len(valid_masks) == 1:
        return valid_masks[0]
    else:
        # Combine masks using logical OR
        combined = valid_masks[0]
        for mask in valid_masks[1:]:
            combined = image.math_img("img1 + img2 > 0", img1=combined, img2=mask)
        return combined

def create_high_events_file(file_path):
    """
    Create events file from high-level behavioral data.
    Returns a dataframe with onset, duration, trial_type columns.
    """
    behav_df = pd.read_csv(file_path)
    
    # Group by block to get block-level events
    # Each block has multiple trials with the same condition (SF)
    events = []
    
    for block_num, block_data in behav_df.groupby('Block'):
        # Get the condition for this block (should be same for all trials in block)
        condition = block_data['Condition'].iloc[0]  # LSF, NSF, or HSF
        
        # Skip blocks that are all oddballs
        non_oddball_trials = block_data[block_data['Oddball'] == False]
        if len(non_oddball_trials) == 0:
            continue
        
        # Get onset (first stimulus onset in block)
        onset = block_data['Stimulus Onset (s)'].min()
        
        # Get offset (last stimulus offset in block)
        offset = block_data['Stimulus Offset (s)'].max()
        
        # Calculate duration
        duration = offset - onset
        
        events.append({
            'onset': onset,
            'duration': duration,
            'trial_type': condition  # LSF, NSF, or HSF
        })
    
    events_df = pd.DataFrame(events)
    
    # Sort by onset to ensure chronological order
    events_df = events_df.sort_values('onset').reset_index(drop=True)
    
    return events_df

def build_design_matrix_for_runs(runs_to_include, runs_to_include_indices, onsets, 
                                   fmri_imgs, confound_files, sub, task):
    """
    Build design matrix for a set of runs.
    Returns design_matrix_allruns and all_fd_scrub.
    """
    design_matrices = []
    all_fd_scrub = []

    for idx, (img, onset_events) in enumerate(zip(fmri_imgs, onsets)):
        # Get TR and number of scans
        fmri_img = image.load_img(img)
        n_scans = fmri_img.shape[-1]
        tr = 2.0  # UPDATE if different
        frame_times = np.arange(n_scans) * tr

        # Load confounds
        confound_df = pd.read_csv(confound_files[idx], sep='\t')
        
        # Calculate FD
        if 'framewise_displacement' in confound_df.columns:
            fd = confound_df['framewise_displacement'].values
        else:
            fd = np.zeros(len(confound_df))
            if len(confound_df) > 1:
                fd[1:] = (
                    np.abs(confound_df['trans_x'].diff()) +
                    np.abs(confound_df['trans_y'].diff()) +
                    np.abs(confound_df['trans_z'].diff()) +
                    np.abs(confound_df['rot_x'].diff() * 50) +
                    np.abs(confound_df['rot_y'].diff() * 50) +
                    np.abs(confound_df['rot_z'].diff() * 50)
                )
        
        # FD scrubbing
        fd_scrub = (fd > fd_threshold).astype(int)
        n_scrubbed = np.sum(fd_scrub)
        
        # Create spike regressors
        fd_regressors = np.zeros((n_scans, n_scrubbed))
        if n_scrubbed > 0:
            scrub_indices = np.where(fd_scrub)[0]
            for i, scrub_idx in enumerate(scrub_indices):
                fd_regressors[scrub_idx, i] = 1
        
        # Select confound variables
        confound_df_subset = confound_df[confound_vars].copy()
        confound_df_subset.fillna(0, inplace=True)
        
        # Make design matrix
        design_matrix = make_first_level_design_matrix(
            frame_times,
            onset_events,
            hrf_model=hrf_model,
            drift_model="polynomial",
            drift_order=3,
            add_regs=confound_df_subset,
            add_reg_names=confound_vars,
            high_pass=high_pass,
        )

        design_matrix = design_matrix.iloc[:, :-1]  # remove constant

        # Add FD spike regressors if enabled
        if APPLY_FD_EXCLUSION and n_scrubbed > 0:
            for i in range(n_scrubbed):
                design_matrix[f'fd_scrub_run{runs_to_include[idx]}_{i}'] = fd_regressors[:, i]
        
        # Run-specific intercepts
        for run_num in range(len(runs_to_include)):
            design_matrix[f'intercept{run_num+1}'] = int(idx == run_num)

        design_matrices.append(design_matrix)
        all_fd_scrub.append(fd_scrub)

    design_matrix_allruns = pd.concat(design_matrices, axis=0, ignore_index=True)
    
    # Clean design matrix
    if design_matrix_allruns.isnull().any().any():
        design_matrix_allruns.fillna(0, inplace=True)
    
    if np.isinf(design_matrix_allruns.values).any():
        design_matrix_allruns = design_matrix_allruns.replace([np.inf, -np.inf], 0)
    
    return design_matrix_allruns, all_fd_scrub

# -----------------------------
# Main Loop
# -----------------------------
patients = subjects[0:30]
controls = subjects[30:]

for sub in subjects:
    print(f"\n{'='*80}")
    print(f"RUNNING SUBJECT {sub}")
    print('='*80)
    
    group = 'Patient' if sub in patients else 'Control'

    # --------------------------------------------------
    # 1. Load V1 Masks from Low-Level Task (T1w space)
    # --------------------------------------------------
    print("\n  Loading V1 masks from low-level task (T1w space)...")
    
    v1_mask_dir = '/Volumes/drive/AVP-BDD/derivatives/functional_v1_masks'
    
    v1_all_path = os.path.join(v1_mask_dir, f'sub-{sub}_functional_V1_mask_all.nii.gz')
    v1_low_sf_path = os.path.join(v1_mask_dir, f'sub-{sub}_functional_V1_mask_low_sf.nii.gz')
    v1_high_sf_path = os.path.join(v1_mask_dir, f'sub-{sub}_functional_V1_mask_high_sf.nii.gz')
    
    v1_masks = {}
    for name, path in [('all', v1_all_path), ('low_sf', v1_low_sf_path), ('high_sf', v1_high_sf_path)]:
        if os.path.exists(path):
            mask = image.load_img(path)
            # Ensure binary
            mask = math_img("img > 0", img=mask)
            v1_masks[name] = mask
            n_voxels = np.sum(mask.get_fdata() > 0)
            print(f"    V1 {name}: {n_voxels} voxels")
        else:
            print(f"    WARNING: V1 {name} mask not found at {path}")
            v1_masks[name] = None

    # --------------------------------------------------
    # 2. Load Categorical ROIs from fLoc (MNI space)
    # --------------------------------------------------
    print("\n  Loading categorical ROIs from fLoc (MNI space)...")
    
    floc_roi_dir = f'/Volumes/drive/AVP-BDD/derivatives/sub-{sub}/ses-01/func'
    
    categorical_rois = {}
    
    # Face ROIs
    face_masks = [load_roi_mask(sub, roi, floc_roi_dir) for roi in FACE_ROIS]
    categorical_rois['face'] = combine_roi_masks(face_masks)
    if categorical_rois['face'] is not None:
        n_voxels = np.sum(categorical_rois['face'].get_fdata() > 0)
        print(f"    Face ROI (combined {FACE_ROIS}): {n_voxels} voxels")
    else:
        print(f"    WARNING: No face ROIs found")
    
    # Body ROIs
    body_masks = [load_roi_mask(sub, roi, floc_roi_dir) for roi in BODY_ROIS]
    categorical_rois['body'] = combine_roi_masks(body_masks)
    if categorical_rois['body'] is not None:
        n_voxels = np.sum(categorical_rois['body'].get_fdata() > 0)
        print(f"    Body ROI (combined {BODY_ROIS}): {n_voxels} voxels")
    else:
        print(f"    WARNING: No body ROIs found")
    
    # House ROIs
    house_masks = [load_roi_mask(sub, roi, floc_roi_dir) for roi in HOUSE_ROIS]
    categorical_rois['house'] = combine_roi_masks(house_masks)
    if categorical_rois['house'] is not None:
        n_voxels = np.sum(categorical_rois['house'].get_fdata() > 0)
        print(f"    House ROI (combined {HOUSE_ROIS}): {n_voxels} voxels")
    else:
        print(f"    WARNING: No house ROIs found")

    # --------------------------------------------------
    # 3. Determine Category-Run Mapping
    # --------------------------------------------------
    print("\n  Determining category-run mapping...")
    
    run_to_category = {}
    category_to_run = {}
    
    for run in runs:
        # Check which category files exist for this run
        for category in CATEGORIES:
            file_path = (
                f"/Volumes/drive/AVP-BDD/behavior/{sub}/high-level/"
                f"Subject_{sub}_Run_{run}_{category}_RT.csv"
            )
            if os.path.exists(file_path):
                run_to_category[run] = category
                category_to_run[category] = run
                print(f"    Run {run}: {category}")
                break
    
    if len(run_to_category) != 3:
        print(f"  WARNING: Could not find all 3 categories for subject {sub}")
        print(f"  Found: {run_to_category}")
        continue
    
    category_run_mapping[sub] = {
        'run_to_category': run_to_category,
        'category_to_run': category_to_run
    }

    # --------------------------------------------------
    # 4. Check Behavioral Performance
    # --------------------------------------------------
    print("\n  Checking behavioral performance...")
    runs_to_include = []
    runs_to_include_indices = []
    categories_in_order = []
    
    for run_idx, run in enumerate(runs):
        category = run_to_category[run]
        file_path = (
            f"/Volumes/drive/AVP-BDD/behavior/{sub}/high-level/"
            f"Subject_{sub}_Run_{run}_{category}_RT.csv"
        )
        
        if not os.path.exists(file_path):
            print(f"    Run {run}: Behavioral file not found, skipping run.")
            behavioral_summary['subject'].append(sub)
            behavioral_summary['run'].append(run)
            behavioral_summary['category'].append(category)
            behavioral_summary['n_oddballs'].append(np.nan)
            behavioral_summary['n_responses'].append(np.nan)
            behavioral_summary['accuracy'].append(np.nan)
            behavioral_summary['excluded'].append(True)
            continue
        
        behav_df = pd.read_csv(file_path)
        n_oddballs = (behav_df['Oddball'] == True).sum()
        n_responses = behav_df['Reaction Time (s)'].notna().sum()
        accuracy = n_responses / n_oddballs if n_oddballs > 0 else 0
        
        if APPLY_BEHAVIORAL_EXCLUSION:
            include_run = accuracy >= accuracy_threshold
        else:
            include_run = True
        
        behavioral_summary['subject'].append(sub)
        behavioral_summary['run'].append(run)
        behavioral_summary['category'].append(category)
        behavioral_summary['n_oddballs'].append(n_oddballs)
        behavioral_summary['n_responses'].append(n_responses)
        behavioral_summary['accuracy'].append(accuracy)
        behavioral_summary['excluded'].append(not include_run)
        
        if include_run:
            runs_to_include.append(run)
            runs_to_include_indices.append(run_idx)
            categories_in_order.append(category)
            print(f"    Run {run} ({category}): {n_responses}/{n_oddballs} oddballs ({accuracy*100:.1f}%) - INCLUDED")
        else:
            print(f"    Run {run} ({category}): {n_responses}/{n_oddballs} oddballs ({accuracy*100:.1f}%) - EXCLUDED")
    
    if len(runs_to_include) == 0:
        print(f"  Subject {sub} has no runs meeting criteria, skipping.")
        continue
    
    print(f"  Including {len(runs_to_include)}/{len(runs)} runs")

    # --------------------------------------------------
    # 5. Load Events
    # --------------------------------------------------
    print("\n  Loading events...")
    onsets = []
    for run, category in zip(runs_to_include, categories_in_order):
        file_path = (
            f"/Volumes/drive/AVP-BDD/behavior/{sub}/high-level/"
            f"Subject_{sub}_Run_{run}_{category}_RT.csv"
        )
        onsets_temp = create_high_events_file(file_path)
        # Add category prefix to trial_type for this run
        onsets_temp['trial_type'] = onsets_temp['trial_type'].apply(lambda x: f'{category}_{x}')
        onsets.append(onsets_temp)
        print(f"    Run {run} ({category}): {len(onsets_temp)} events")

    # --------------------------------------------------
    # 6. Load Confounds (same for both GLMs)
    # --------------------------------------------------
    all_confound_files = layout.get(
        subject=sub,
        datatype='func',
        task=task,
        desc='confounds',
        extension="tsv",
        return_type='file'
    )
    
    confound_files = [all_confound_files[idx] for idx in runs_to_include_indices]

    # --------------------------------------------------
    # 7. GLM #1: V1 ROIs (T1w space)
    # --------------------------------------------------
    print("\n" + "="*80)
    print("GLM #1: V1 ROIs (T1w space)")
    print("="*80)
    
    # Load fMRI images in T1w space
    all_fmri_imgs_t1w = layout.get(
        subject=sub,
        datatype='func',
        task=task,
        desc='preproc',
        space='T1w',
        extension='nii.gz',
        return_type='file'
    )
    
    fmri_imgs_t1w = [all_fmri_imgs_t1w[idx] for idx in runs_to_include_indices]
    print(f"  Loaded {len(fmri_imgs_t1w)} fMRI images in T1w space")
    
    # Build design matrix
    design_matrix_t1w, all_fd_scrub = build_design_matrix_for_runs(
        runs_to_include, runs_to_include_indices, onsets, 
        fmri_imgs_t1w, confound_files, sub, task
    )
    
    print(f"  Design matrix shape: {design_matrix_t1w.shape}")
    
    # Report FD scrubbing (only once)
    total_scrubbed = sum([np.sum(fd) for fd in all_fd_scrub])
    total_frames = len(design_matrix_t1w)
    pct_scrubbed = 100 * total_scrubbed / total_frames
    print(f"  FD scrubbing: {total_scrubbed}/{total_frames} frames ({pct_scrubbed:.1f}%)")
    
    fd_summary['subject'].append(sub)
    fd_summary['total_frames'].append(total_frames)
    fd_summary['scrubbed_frames'].append(total_scrubbed)
    fd_summary['pct_scrubbed'].append(pct_scrubbed)
    
    # Initialize subject results
    subject_results = {
        'subject': sub,
        'group': group,
        'v1_all': {},
        'v1_low_sf': {},
        'v1_high_sf': {},
        'face_roi': {},
        'body_roi': {},
        'house_roi': {}
    }
    
    # Run GLM for V1 ROIs
    v1_roi_dict = {
        'v1_all': v1_masks.get('all'),
        'v1_low_sf': v1_masks.get('low_sf'),
        'v1_high_sf': v1_masks.get('high_sf')
    }
    
    for roi_name, roi_mask in v1_roi_dict.items():
        if roi_mask is None:
            print(f"  Skipping {roi_name} (mask not available)")
            continue
        
        print(f"  Processing {roi_name}...")
        
        # Extract time series
        masker = NiftiMasker(mask_img=roi_mask, standardize=False)
        
        time_series = []
        for img in fmri_imgs_t1w:
            time_series.append(masker.fit_transform(img))
        time_series = np.concatenate(time_series, axis=0)
        
        n_voxels = time_series.shape[1]
        print(f"    Time series shape: {time_series.shape} ({n_voxels} voxels)")
        
        # Run GLM
        labels, estimates = run_glm(time_series, design_matrix_t1w.values)
        
        # Build contrast vectors
        contrast_matrix = np.eye(design_matrix_t1w.shape[1])
        basic_contrasts = {
            column: contrast_matrix[i]
            for i, column in enumerate(design_matrix_t1w.columns)
        }
        
        # Get all condition names
        condition_names = []
        for cat in CATEGORIES:
            for sf in SPATIAL_FREQUENCIES:
                condition_name = f'{cat}_{sf}'
                if condition_name in design_matrix_t1w.columns:
                    condition_names.append(condition_name)
        
        # Compute contrasts
        for condition in condition_names:
            if condition not in basic_contrasts:
                continue
            
            contrast_vec = basic_contrasts[condition]
            contrast = compute_contrast(labels, estimates, contrast_vec)
            
            beta_vox = contrast.effect_size()
            z_vox = contrast.z_score()
            
            # Calculate statistics
            mean_beta = np.mean(beta_vox)
            std_beta = np.std(beta_vox)
            median_beta = np.median(beta_vox)
            mean_z = np.mean(z_vox)
            
            # Positive voxels
            positive_mask = beta_vox > 0
            n_positive = np.sum(positive_mask)
            pct_positive = 100 * n_positive / len(beta_vox)
            
            subject_results[roi_name][condition] = {
                "mean_beta": mean_beta,
                "std_beta": std_beta,
                "median_beta": median_beta,
                "mean_z": mean_z,
                "n_positive_voxels": n_positive,
                "pct_positive_voxels": pct_positive,
                "n_voxels": n_voxels
            }

    # --------------------------------------------------
    # 8. GLM #2: Categorical ROIs (MNI space)
    # --------------------------------------------------
    print("\n" + "="*80)
    print("GLM #2: Categorical ROIs (MNI space)")
    print("="*80)
    
    # Load fMRI images in MNI space
    all_fmri_imgs_mni = layout.get(
        subject=sub,
        datatype='func',
        task=task,
        desc='preproc',
        space='MNI152NLin2009cAsym',
        extension='nii.gz',
        return_type='file'
    )
    
    fmri_imgs_mni = [all_fmri_imgs_mni[idx] for idx in runs_to_include_indices]
    print(f"  Loaded {len(fmri_imgs_mni)} fMRI images in MNI space")
    
    # Build design matrix (same as T1w GLM)
    design_matrix_mni, _ = build_design_matrix_for_runs(
        runs_to_include, runs_to_include_indices, onsets, 
        fmri_imgs_mni, confound_files, sub, task
    )
    
    print(f"  Design matrix shape: {design_matrix_mni.shape}")
    
    # Run GLM for categorical ROIs
    cat_roi_dict = {
        'face_roi': categorical_rois.get('face'),
        'body_roi': categorical_rois.get('body'),
        'house_roi': categorical_rois.get('house')
    }
    
    for roi_name, roi_mask in cat_roi_dict.items():
        if roi_mask is None:
            print(f"  Skipping {roi_name} (mask not available)")
            continue
        
        print(f"  Processing {roi_name}...")
        
        # Extract time series
        masker = NiftiMasker(mask_img=roi_mask, standardize=False)
        
        time_series = []
        for img in fmri_imgs_mni:
            time_series.append(masker.fit_transform(img))
        time_series = np.concatenate(time_series, axis=0)
        
        n_voxels = time_series.shape[1]
        print(f"    Time series shape: {time_series.shape} ({n_voxels} voxels)")
        
        # Run GLM
        labels, estimates = run_glm(time_series, design_matrix_mni.values)
        
        # Build contrast vectors
        contrast_matrix = np.eye(design_matrix_mni.shape[1])
        basic_contrasts = {
            column: contrast_matrix[i]
            for i, column in enumerate(design_matrix_mni.columns)
        }
        
        # Get all condition names
        condition_names = []
        for cat in CATEGORIES:
            for sf in SPATIAL_FREQUENCIES:
                condition_name = f'{cat}_{sf}'
                if condition_name in design_matrix_mni.columns:
                    condition_names.append(condition_name)
        
        # Compute contrasts
        for condition in condition_names:
            if condition not in basic_contrasts:
                continue
            
            contrast_vec = basic_contrasts[condition]
            contrast = compute_contrast(labels, estimates, contrast_vec)
            
            beta_vox = contrast.effect_size()
            z_vox = contrast.z_score()
            
            # Calculate statistics
            mean_beta = np.mean(beta_vox)
            std_beta = np.std(beta_vox)
            median_beta = np.median(beta_vox)
            mean_z = np.mean(z_vox)
            
            # Positive voxels
            positive_mask = beta_vox > 0
            n_positive = np.sum(positive_mask)
            pct_positive = 100 * n_positive / len(beta_vox)
            
            subject_results[roi_name][condition] = {
                "mean_beta": mean_beta,
                "std_beta": std_beta,
                "median_beta": median_beta,
                "mean_z": mean_z,
                "n_positive_voxels": n_positive,
                "pct_positive_voxels": pct_positive,
                "n_voxels": n_voxels
            }
    
    # Store results
    all_subject_results[sub] = subject_results
    
    # Add to summary
    glm_summary['subject'].append(sub)
    glm_summary['group'].append(group)
    glm_summary['n_v1_voxels'].append(subject_results.get('v1_all', {}).get(condition_names[0] if condition_names else 'Face_LSF', {}).get('n_voxels', 0))
    glm_summary['n_v1_low_sf_voxels'].append(subject_results.get('v1_low_sf', {}).get(condition_names[0] if condition_names else 'Face_LSF', {}).get('n_voxels', 0))
    glm_summary['n_v1_high_sf_voxels'].append(subject_results.get('v1_high_sf', {}).get(condition_names[0] if condition_names else 'Face_LSF', {}).get('n_voxels', 0))
    glm_summary['n_face_roi_voxels'].append(subject_results.get('face_roi', {}).get(condition_names[0] if condition_names else 'Face_LSF', {}).get('n_voxels', 0))
    glm_summary['n_body_roi_voxels'].append(subject_results.get('body_roi', {}).get(condition_names[0] if condition_names else 'Face_LSF', {}).get('n_voxels', 0))
    glm_summary['n_house_roi_voxels'].append(subject_results.get('house_roi', {}).get(condition_names[0] if condition_names else 'Face_LSF', {}).get('n_voxels', 0))
    
    # --------------------------------------------------
    # 9. Create Visualization
    # --------------------------------------------------
    print("\n  Creating visualization...")
    
    os.makedirs('/Volumes/drive/AVP-BDD/results/high-level/glm_results/individual_subjects', exist_ok=True)
    
    # Create summary figure showing mean betas for each ROI and condition
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle(f'Subject {sub} ({group}): High-Level Task GLM Results', fontsize=16, fontweight='bold')
    
    roi_names_plot = ['v1_all', 'v1_low_sf', 'v1_high_sf', 'face_roi', 'body_roi', 'house_roi']
    roi_labels = {
        'v1_all': 'V1 (All) [T1w]',
        'v1_low_sf': 'V1 (Low-SF) [T1w]',
        'v1_high_sf': 'V1 (High-SF) [T1w]',
        'face_roi': 'Face ROI [MNI]',
        'body_roi': 'Body ROI [MNI]',
        'house_roi': 'House ROI [MNI]'
    }
    
    for idx, roi_name in enumerate(roi_names_plot):
        ax = axes[idx // 3, idx % 3]
        
        if roi_name not in subject_results or not subject_results[roi_name]:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', fontsize=14)
            ax.set_title(roi_labels[roi_name])
            ax.axis('off')
            continue
        
        # Extract data for this ROI
        conditions = sorted([k for k in subject_results[roi_name].keys()])
        means = [subject_results[roi_name][c]['mean_beta'] for c in conditions]
        stds = [subject_results[roi_name][c]['std_beta'] for c in conditions]
        
        # Bar plot
        x = np.arange(len(conditions))
        ax.bar(x, means, yerr=stds, capsize=5, alpha=0.7, edgecolor='black', linewidth=1.5)
        ax.set_xticks(x)
        ax.set_xticklabels(conditions, rotation=45, ha='right', fontsize=9)
        ax.set_ylabel('Beta Value', fontweight='bold')
        ax.set_title(roi_labels[roi_name], fontweight='bold')
        ax.axhline(0, color='black', linestyle='--', linewidth=1)
        ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    save_path = f'/Volumes/drive/AVP-BDD/results/high-level/glm_results/individual_subjects/sub-{sub}_glm_summary.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"  Saved: {save_path}")

# Add this import at the top with the other imports
from nilearn.plotting import plot_glass_brain, plot_stat_map
from nilearn.image import new_img_like

# [Keep all the previous code the same until after the main subject loop ends]
# Add this section after the main subject loop completes:

# --------------------------------------------------
# CREATE GROUP-LEVEL VISUALIZATIONS
# --------------------------------------------------
print("\n" + "="*80)
print("CREATING GROUP-LEVEL VISUALIZATIONS")
print("="*80)

os.makedirs('/Volumes/drive/AVP-BDD/results/high-level/glm_results/group_visualizations', exist_ok=True)

def create_group_barplot(all_subject_results, roi_name, contrast_names, group_filter=None, title_suffix=""):
    """
    Create bar plot showing mean ± SEM across subjects for specified contrasts.
    """
    if group_filter == 'patient':
        subjects_to_include = [s for s in all_subject_results.keys() if s in patients]
        color = '#f5576c'
    elif group_filter == 'control':
        subjects_to_include = [s for s in all_subject_results.keys() if s in controls]
        color = '#4facfe'
    else:
        subjects_to_include = list(all_subject_results.keys())
        color = '#7f77dd'
    
    # Extract data
    data = {contrast: [] for contrast in contrast_names}
    
    for sub in subjects_to_include:
        if roi_name not in all_subject_results[sub]:
            continue
        roi_data = all_subject_results[sub][roi_name]
        for contrast in contrast_names:
            if contrast in roi_data:
                data[contrast].append(roi_data[contrast]['mean_beta'])
    
    # Calculate means and SEMs
    means = []
    sems = []
    labels = []
    
    for contrast in contrast_names:
        if len(data[contrast]) > 0:
            means.append(np.mean(data[contrast]))
            sems.append(np.std(data[contrast]) / np.sqrt(len(data[contrast])))
            labels.append(contrast)
    
    if len(means) == 0:
        return None
    
    # Create plot
    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(labels))
    ax.bar(x, means, yerr=sems, capsize=5, alpha=0.7, color=color, 
           edgecolor='black', linewidth=1.5)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=10)
    ax.set_ylabel('Mean Beta Value', fontsize=12, fontweight='bold')
    ax.set_title(f'{roi_name.upper()} - {title_suffix}\nN={len(subjects_to_include)}', 
                 fontsize=14, fontweight='bold')
    ax.axhline(0, color='black', linestyle='--', linewidth=1)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    
    # Save to bytes
    from io import BytesIO
    buf = BytesIO()
    plt.savefig(buf, format='png', dpi=150, bbox_inches='tight')
    buf.seek(0)
    img_b64 = base64.b64encode(buf.read()).decode('utf-8')
    plt.close()
    
    return img_b64

# Create group visualizations for each ROI
group_viz_data = {
    'v1_all': {},
    'face_roi': {},
    'body_roi': {},
    'house_roi': {}
}

# Define key contrasts to visualize
individual_conditions = [f'{cat}_{sf}' for cat in CATEGORIES for sf in SPATIAL_FREQUENCIES]
within_category_contrasts = [f'{cat}_{c}' for cat in CATEGORIES for c in ['LSF-NSF', 'HSF-LSF']]
pooled_contrasts = ['AllCategories_LSF', 'AllCategories_NSF', 'AllCategories_HSF',
                    'AllCategories_LSF-NSF', 'AllCategories_HSF-NSF']

for roi_name in ['v1_all', 'face_roi', 'body_roi', 'house_roi']:
    print(f"\n  Creating visualizations for {roi_name}...")
    
    # All subjects
    group_viz_data[roi_name]['all_conditions_all'] = create_group_barplot(
        all_subject_results, roi_name, individual_conditions, 
        group_filter=None, title_suffix="All Conditions (All Subjects)"
    )
    group_viz_data[roi_name]['all_conditions_patients'] = create_group_barplot(
        all_subject_results, roi_name, individual_conditions,
        group_filter='patient', title_suffix="All Conditions (Patients)"
    )
    group_viz_data[roi_name]['all_conditions_controls'] = create_group_barplot(
        all_subject_results, roi_name, individual_conditions,
        group_filter='control', title_suffix="All Conditions (Controls)"
    )
    
    # Pooled contrasts
    group_viz_data[roi_name]['pooled_all'] = create_group_barplot(
        all_subject_results, roi_name, pooled_contrasts,
        group_filter=None, title_suffix="Pooled SF Contrasts (All Subjects)"
    )
    group_viz_data[roi_name]['pooled_patients'] = create_group_barplot(
        all_subject_results, roi_name, pooled_contrasts,
        group_filter='patient', title_suffix="Pooled SF Contrasts (Patients)"
    )
    group_viz_data[roi_name]['pooled_controls'] = create_group_barplot(
        all_subject_results, roi_name, pooled_contrasts,
        group_filter='control', title_suffix="Pooled SF Contrasts (Controls)"
    )

print("  Group visualizations complete")

# --------------------------------------------------
# CREATE EMBEDDED HTML QC REPORT
# --------------------------------------------------
print("\n" + "="*80)
print("CREATING HTML QC REPORT")
print("="*80)

def create_html_report(all_subject_results, group_viz_data, output_path):
    """
    Create comprehensive HTML QC report with embedded images.
    """
    timestamp = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    n_subjects = len(all_subject_results)
    n_patients = len([s for s in all_subject_results.keys() if s in patients])
    n_controls = len([s for s in all_subject_results.keys() if s in controls])
    
    # Build subject summary table
    table_rows = ""
    for sub, results in all_subject_results.items():
        group = results['group']
        group_class = 'grp-patient' if group == 'Patient' else 'grp-control'
        
        # Get voxel counts
        n_v1 = results.get('v1_all', {}).get('Face_LSF', {}).get('n_voxels', 0)
        n_face = results.get('face_roi', {}).get('Face_LSF', {}).get('n_voxels', 0)
        n_body = results.get('body_roi', {}).get('Body_LSF', {}).get('n_voxels', 0)
        n_house = results.get('house_roi', {}).get('House_LSF', {}).get('n_voxels', 0)
        
        # Get sample beta values (AllCategories_LSF from V1)
        v1_lsf_beta = results.get('v1_all', {}).get('AllCategories_LSF', {}).get('mean_beta', np.nan)
        
        table_rows += f"""
        <tr data-sub="sub-{sub}" data-grp="{group}">
            <td class="sub-cell">sub-{sub}</td>
            <td><span class="grp {group_class}">{group}</span></td>
            <td class="num-cell">{n_v1}</td>
            <td class="num-cell">{n_face}</td>
            <td class="num-cell">{n_body}</td>
            <td class="num-cell">{n_house}</td>
            <td class="num-cell">{v1_lsf_beta:.4f}</td>
        </tr>
        """
    
    # Build individual subject detail panels
    detail_panels = ""
    for sub, results in all_subject_results.items():
        group = results['group']
        group_class = 'grp-patient' if group == 'Patient' else 'grp-control'
        
        # Load the subject's visualization
        img_path = f'/Volumes/drive/AVP-BDD/results/high-level/glm_results/individual_subjects/sub-{sub}_glm_summary.png'
        img_b64 = image_to_base64(img_path)
        
        img_html = ""
        if img_b64:
            img_html = f'<img src="data:image/png;base64,{img_b64}" style="width:100%;border-radius:8px;">'
        else:
            img_html = '<div class="no-img">Image not available</div>'
        
        # Summary statistics
        stats_html = ""
        for roi_name in ['v1_all', 'face_roi', 'body_roi', 'house_roi']:
            if roi_name not in results or not results[roi_name]:
                continue
            
            roi_label = {'v1_all': 'V1 (T1w)', 'face_roi': 'Face ROI (MNI)', 
                        'body_roi': 'Body ROI (MNI)', 'house_roi': 'House ROI (MNI)'}[roi_name]
            
            # Get key contrasts
            lsf_all = results[roi_name].get('AllCategories_LSF', {}).get('mean_beta', np.nan)
            nsf_all = results[roi_name].get('AllCategories_NSF', {}).get('mean_beta', np.nan)
            hsf_all = results[roi_name].get('AllCategories_HSF', {}).get('mean_beta', np.nan)
            lsf_nsf_diff = results[roi_name].get('AllCategories_LSF-NSF', {}).get('mean_beta', np.nan)
            
            stats_html += f"""
            <div class="stats-card">
                <div class="stats-title">{roi_label}</div>
                <div class="stats-grid">
                    <div class="stat-item">
                        <span class="stat-label">LSF (pooled)</span>
                        <span class="stat-val">{lsf_all:.4f}</span>
                    </div>
                    <div class="stat-item">
                        <span class="stat-label">NSF (pooled)</span>
                        <span class="stat-val">{nsf_all:.4f}</span>
                    </div>
                    <div class="stat-item">
                        <span class="stat-label">HSF (pooled)</span>
                        <span class="stat-val">{hsf_all:.4f}</span>
                    </div>
                    <div class="stat-item">
                        <span class="stat-label">LSF - NSF</span>
                        <span class="stat-val">{lsf_nsf_diff:.4f}</span>
                    </div>
                </div>
            </div>
            """
        
        detail_panels += f"""
        <div class="detail-panel" id="panel-sub-{sub}" style="display:none;">
            <div class="detail-header">
                <span class="detail-title">sub-{sub}</span>
                <span class="grp {group_class}">{group}</span>
            </div>
            <div class="stats-row">{stats_html}</div>
            <div class="img-section">
                <div class="img-section-title">Individual Condition Responses</div>
                {img_html}
            </div>
        </div>
        """
    
    # Build group-level panels
    group_panels = ""
    
    for roi_name in ['v1_all', 'face_roi', 'body_roi', 'house_roi']:
        roi_label = {'v1_all': 'V1 (T1w)', 'face_roi': 'Face ROI (MNI)', 
                    'body_roi': 'Body ROI (MNI)', 'house_roi': 'House ROI (MNI)'}[roi_name]
        
        # Get images
        all_cond_all = group_viz_data[roi_name].get('all_conditions_all')
        all_cond_pat = group_viz_data[roi_name].get('all_conditions_patients')
        all_cond_ctrl = group_viz_data[roi_name].get('all_conditions_controls')
        pooled_all = group_viz_data[roi_name].get('pooled_all')
        pooled_pat = group_viz_data[roi_name].get('pooled_patients')
        pooled_ctrl = group_viz_data[roi_name].get('pooled_controls')
        
        def img_tag(b64):
            if b64:
                return f'<img src="data:image/png;base64,{b64}" style="width:100%;border-radius:8px;">'
            return '<div class="no-img">No data</div>'
        
        group_panels += f"""
        <div class="group-panel" id="group-{roi_name}" style="display:{'block' if roi_name == 'v1_all' else 'none'};">
            <h3>{roi_label}</h3>
            
            <div class="group-section">
                <div class="group-section-title">Individual Conditions</div>
                <div class="group-grid">
                    <div class="group-img-wrap">
                        <div class="group-img-label">All Subjects (N={n_subjects})</div>
                        {img_tag(all_cond_all)}
                    </div>
                    <div class="group-img-wrap">
                        <div class="group-img-label">Patients (N={n_patients})</div>
                        {img_tag(all_cond_pat)}
                    </div>
                    <div class="group-img-wrap">
                        <div class="group-img-label">Controls (N={n_controls})</div>
                        {img_tag(all_cond_ctrl)}
                    </div>
                </div>
            </div>
            
            <div class="group-section">
                <div class="group-section-title">Pooled SF Contrasts</div>
                <div class="group-grid">
                    <div class="group-img-wrap">
                        <div class="group-img-label">All Subjects (N={n_subjects})</div>
                        {img_tag(pooled_all)}
                    </div>
                    <div class="group-img-wrap">
                        <div class="group-img-label">Patients (N={n_patients})</div>
                        {img_tag(pooled_pat)}
                    </div>
                    <div class="group-img-wrap">
                        <div class="group-img-label">Controls (N={n_controls})</div>
                        {img_tag(pooled_ctrl)}
                    </div>
                </div>
            </div>
        </div>
        """
    
    # Generate HTML
    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>High-Level Task GLM QC Report</title>
<style>
:root {{
    --bg:#111;--surface:#1c1c1e;--surface2:#252528;--border:#2e2e32;
    --text:#e8e8ea;--muted:#8e8e93;--accent:#7f77dd;
    --patient:#f5576c;--control:#4facfe;
}}
*{{box-sizing:border-box;margin:0;padding:0}}
body{{background:var(--bg);color:var(--text);font-family:ui-monospace,'SF Mono',monospace;font-size:13px}}
.page{{max-width:1600px;margin:0 auto;padding:2rem}}
h1{{font-size:18px;font-weight:500;margin-bottom:.25rem}}
.subtitle{{color:var(--muted);font-size:12px;margin-bottom:1.5rem}}

.cards{{display:flex;gap:12px;margin-bottom:1.5rem;flex-wrap:wrap}}
.card{{background:var(--surface);border:.5px solid var(--border);border-radius:10px;padding:1rem 1.25rem;flex:1;min-width:120px}}
.card-label{{font-size:11px;color:var(--muted);margin-bottom:4px}}
.card-val{{font-size:22px;font-weight:500}}

.section-title{{font-size:12px;color:var(--muted);text-transform:uppercase;letter-spacing:.08em;margin:1.5rem 0 .75rem}}

.filter-row{{display:flex;gap:8px;margin-bottom:.75rem;flex-wrap:wrap}}
.filter-btn{{background:var(--surface2);border:.5px solid var(--border);color:var(--muted);border-radius:6px;padding:4px 12px;font-size:12px;cursor:pointer;font-family:inherit}}
.filter-btn.active{{border-color:var(--accent);color:var(--accent)}}

.table-wrap{{overflow-x:auto;border-radius:10px;border:.5px solid var(--border);margin-bottom:1rem}}
table{{width:100%;border-collapse:collapse}}
th{{background:var(--surface2);color:var(--muted);font-size:11px;font-weight:500;padding:8px 10px;text-align:left;border-bottom:.5px solid var(--border)}}
tr{{border-bottom:.5px solid var(--border);cursor:pointer;transition:background .1s}}
tr:hover{{background:var(--surface2)}}tr.selected{{background:#2a2a40}}
td{{padding:7px 10px}}
.sub-cell{{font-weight:500}}
.num-cell{{text-align:right;font-family:ui-monospace}}

.grp{{display:inline-block;font-size:11px;padding:2px 8px;border-radius:10px;font-weight:500}}
.grp-patient{{background:#3d1a0a;color:#f5576c}}
.grp-control{{background:#0c1f3d;color:#4facfe}}

.nav-row{{display:flex;gap:8px;margin-bottom:1rem}}
.nav-btn{{background:var(--surface2);border:.5px solid var(--border);color:var(--text);border-radius:6px;padding:5px 16px;font-size:12px;cursor:pointer;font-family:inherit}}
.nav-btn:hover{{border-color:var(--accent);color:var(--accent)}}
.nav-info{{font-size:12px;color:var(--muted);margin-left:auto}}

.detail-panel{{background:var(--surface);border:.5px solid var(--border);border-radius:10px;margin-top:1rem;padding:1.25rem}}
.detail-header{{display:flex;align-items:center;gap:10px;margin-bottom:1rem}}
.detail-title{{font-size:15px;font-weight:500}}

.stats-row{{display:grid;grid-template-columns:repeat(auto-fit,minmax(280px,1fr));gap:12px;margin-bottom:1rem}}
.stats-card{{background:var(--surface2);border:.5px solid var(--border);border-radius:8px;padding:12px}}
.stats-title{{font-size:12px;font-weight:500;color:var(--accent);margin-bottom:8px}}
.stats-grid{{display:grid;grid-template-columns:1fr 1fr;gap:8px}}
.stat-item{{display:flex;flex-direction:column}}
.stat-label{{font-size:10px;color:var(--muted)}}
.stat-val{{font-size:12px;font-family:ui-monospace}}

.img-section{{margin-top:1rem}}
.img-section-title{{font-size:12px;color:var(--muted);margin-bottom:8px}}
.no-img{{background:var(--bg);border-radius:8px;padding:2rem;text-align:center;color:var(--muted)}}

.roi-tabs{{display:flex;gap:4px;margin-bottom:1rem}}
.roi-tab{{background:var(--surface2);border:.5px solid var(--border);color:var(--muted);border-radius:6px;padding:4px 14px;font-size:12px;cursor:pointer;font-family:inherit}}
.roi-tab.active{{border-color:var(--accent);color:var(--accent)}}

.group-panel{{background:var(--surface);border:.5px solid var(--border);border-radius:10px;padding:1.25rem;margin-top:1rem}}
.group-panel h3{{font-size:16px;margin-bottom:1rem}}
.group-section{{margin-bottom:2rem}}
.group-section-title{{font-size:13px;font-weight:500;color:var(--accent);margin-bottom:12px}}
.group-grid{{display:grid;grid-template-columns:repeat(auto-fit,minmax(400px,1fr));gap:16px}}
.group-img-wrap{{}}
.group-img-label{{font-size:11px;color:var(--muted);margin-bottom:6px}}
</style>
</head>
<body>
<div class="page">
<h1>High-Level Task GLM QC Report</h1>
<div class="subtitle">Generated {timestamp} • {n_subjects} subjects • Fully embedded</div>

<div class="cards">
    <div class="card"><div class="card-label">Total Subjects</div><div class="card-val">{n_subjects}</div></div>
    <div class="card"><div class="card-label">Patients</div><div class="card-val" style="color:var(--patient)">{n_patients}</div></div>
    <div class="card"><div class="card-label">Controls</div><div class="card-val" style="color:var(--control)">{n_controls}</div></div>
    <div class="card"><div class="card-label">ROIs Analyzed</div><div class="card-val">4</div></div>
    <div class="card"><div class="card-label">Contrasts per ROI</div><div class="card-val">20</div></div>
</div>

<div class="section-title">Group-Level Results</div>
<div class="roi-tabs">
    <button class="roi-tab active" onclick="switchROI('v1_all',this)">V1 (T1w)</button>
    <button class="roi-tab" onclick="switchROI('face_roi',this)">Face ROI (MNI)</button>
    <button class="roi-tab" onclick="switchROI('body_roi',this)">Body ROI (MNI)</button>
    <button class="roi-tab" onclick="switchROI('house_roi',this)">House ROI (MNI)</button>
</div>
<div id="group-area">{group_panels}</div>

<div class="section-title">Individual Subjects — Click row to expand</div>
<div class="filter-row">
    <button class="filter-btn active" onclick="filterTable('all',this)">All</button>
    <button class="filter-btn" onclick="filterTable('Patient',this)">Patients</button>
    <button class="filter-btn" onclick="filterTable('Control',this)">Controls</button>
</div>

<div class="table-wrap">
    <table id="main-table">
        <thead>
            <tr>
                <th>Subject</th>
                <th>Group</th>
                <th>V1 voxels</th>
                <th>Face voxels</th>
                <th>Body voxels</th>
                <th>House voxels</th>
                <th>V1 LSF β</th>
            </tr>
        </thead>
        <tbody>{table_rows}</tbody>
    </table>
</div>

<div class="nav-row">
    <button class="nav-btn" onclick="navSubject(-1)">← Prev</button>
    <button class="nav-btn" onclick="navSubject(1)">Next →</button>
    <span class="nav-info" id="nav-info">Click a row to begin</span>
</div>

<div id="detail-area">{detail_panels}</div>
</div>

<script>
const allRows = Array.from(document.querySelectorAll('#main-table tbody tr'));
let visibleRows = allRows.slice();
let activeIdx = null;

function getPanel(sub) {{ return document.getElementById('panel-' + sub); }}

function showSubject(idx) {{
    if (activeIdx !== null) {{
        const prev = visibleRows[activeIdx];
        if (prev) {{
            prev.classList.remove('selected');
            const p = getPanel(prev.dataset.sub);
            if (p) p.style.display = 'none';
        }}
    }}
    if (idx < 0 || idx >= visibleRows.length) return;
    activeIdx = idx;
    const row = visibleRows[idx];
    row.classList.add('selected');
    row.scrollIntoView({{ behavior: 'smooth', block: 'nearest' }});
    const panel = getPanel(row.dataset.sub);
    if (panel) {{
        panel.style.display = 'block';
        panel.scrollIntoView({{ behavior: 'smooth', block: 'nearest' }});
    }}
    document.getElementById('nav-info').textContent = 
        row.dataset.sub + ' (' + (idx + 1) + ' of ' + visibleRows.length + ')';
}}

allRows.forEach(row => {{
    row.addEventListener('click', () => {{
        const idx = visibleRows.indexOf(row);
        if (idx === -1) return;
        if (activeIdx === idx) {{
            row.classList.remove('selected');
            const p = getPanel(row.dataset.sub);
            if (p) p.style.display = 'none';
            activeIdx = null;
            document.getElementById('nav-info').textContent = 'Click a row to begin';
        }} else {{
            showSubject(idx);
        }}
    }});
}});

function navSubject(dir) {{
    if (activeIdx === null) {{ showSubject(0); return; }}
    const next = activeIdx + dir;
    if (next >= 0 && next < visibleRows.length) showSubject(next);
}}

function filterTable(type, btn) {{
    document.querySelectorAll('.filter-btn').forEach(b => b.classList.remove('active'));
    btn.classList.add('active');
    if (activeIdx !== null) {{
        const row = visibleRows[activeIdx];
        if (row) {{
            row.classList.remove('selected');
            const p = getPanel(row.dataset.sub);
            if (p) p.style.display = 'none';
        }}
        activeIdx = null;
        document.getElementById('nav-info').textContent = 'Click a row to begin';
    }}
    allRows.forEach(row => {{
        const show = type === 'all' || row.dataset.grp === type;
        row.style.display = show ? '' : 'none';
    }});
    visibleRows = allRows.filter(r => r.style.display !== 'none');
}}

function switchROI(roiName, btn) {{
    document.querySelectorAll('.roi-tab').forEach(b => b.classList.remove('active'));
    btn.classList.add('active');
    document.querySelectorAll('.group-panel').forEach(p => {{
        p.style.display = p.id === 'group-' + roiName ? 'block' : 'none';
    }});
}}

document.addEventListener('keydown', e => {{
    if (e.key === 'ArrowRight') navSubject(1);
    if (e.key === 'ArrowLeft') navSubject(-1);
}});
</script>
</body>
</html>"""
    
    with open(output_path, 'w') as f:
        f.write(html)
    
    print(f"  HTML report saved: {output_path}")

# Generate the HTML report
html_output_path = f'/Volumes/drive/AVP-BDD/results/high-level/glm_results/QC_report_high_level_glm_{timestamp}.html'
create_html_report(all_subject_results, group_viz_data, html_output_path)

# --------------------------------------------------
# Save Results
# --------------------------------------------------
print("\n" + "="*80)
print("SAVING RESULTS")
print("="*80)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# Save pickle of all results
os.makedirs('/Volumes/drive/AVP-BDD/results/high-level/glm_results', exist_ok=True)
results_path = f'/Volumes/drive/AVP-BDD/results/high-level/glm_results/all_subject_glm_results_{timestamp}.pkl'
with open(results_path, 'wb') as f:
    pickle.dump(all_subject_results, f)
print(f"\n✅ All GLM results saved: {results_path}")

# Save category-run mapping
category_mapping_df = []
for sub, mapping in category_run_mapping.items():
    for run, category in mapping['run_to_category'].items():
        category_mapping_df.append({
            'subject': sub,
            'run': run,
            'category': category
        })
pd.DataFrame(category_mapping_df).to_csv(
    f'/Volumes/drive/AVP-BDD/results/high-level/glm_results/category_run_mapping_{timestamp}.csv',
    index=False
)

# Save summaries as CSV
fd_df = pd.DataFrame(fd_summary)
behav_df = pd.DataFrame(behavioral_summary)
glm_df = pd.DataFrame(glm_summary)

fd_df.to_csv(f'/Volumes/drive/AVP-BDD/results/high-level/glm_results/fd_summary_{timestamp}.csv', index=False)
behav_df.to_csv(f'/Volumes/drive/AVP-BDD/results/high-level/glm_results/behavioral_summary_{timestamp}.csv', index=False)
glm_df.to_csv(f'/Volumes/drive/AVP-BDD/results/high-level/glm_results/glm_summary_{timestamp}.csv', index=False)

print(f"✅ Summary CSVs saved")

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)
print(f"\nTotal subjects analyzed: {len(all_subject_results)}")
print(f"\nAll results saved to: /Volumes/drive/AVP-BDD/results/high-level/glm_results/")

BIDS Layout: ...umes/drive/AVP-BDD/derivatives | Subjects: 59 | Sessions: 59 | Runs: 177

RUNNING SUBJECT 102

  Loading V1 masks from low-level task (T1w space)...
    V1 all: 2212 voxels
    V1 low_sf: 1918 voxels
    V1 high_sf: 2189 voxels

  Loading categorical ROIs from fLoc (MNI space)...
    Face ROI (combined ['FFA', 'OFA', 'FuS']): 147 voxels
    Body ROI (combined ['EBA', 'FBA', 'G_S_occ_inf']): 245 voxels
    House ROI (combined ['PPA', 'RSC', 'TOS']): 166 voxels

  Determining category-run mapping...
    Run 1: House
    Run 2: Face
    Run 3: Body

  Checking behavioral performance...
    Run 1 (House): 27/27 oddballs (100.0%) - INCLUDED
    Run 2 (Face): 37/37 oddballs (100.0%) - INCLUDED
    Run 3 (Body): 32/32 oddballs (100.0%) - INCLUDED
  Including 3/3 runs

  Loading events...
    Run 1 (House): 15 events
    Run 2 (Face): 15 events
    Run 3 (Body): 15 events

GLM #1: V1 ROIs (T1w space)
  Loaded 3 fMRI images in T1w space


/var/folders/hq/ms5zk3kj1pd2kwh1bnvtcr0m0000gn/T/ipykernel_12338/1092442517.py:219: UserWarning: Matrix is singular at working precision, regularizing...
  design_matrix = make_first_level_design_matrix(
/var/folders/hq/ms5zk3kj1pd2kwh1bnvtcr0m0000gn/T/ipykernel_12338/1092442517.py:219: UserWarning: Matrix is singular at working precision, regularizing...
  design_matrix = make_first_level_design_matrix(
/var/folders/hq/ms5zk3kj1pd2kwh1bnvtcr0m0000gn/T/ipykernel_12338/1092442517.py:219: UserWarning: Matrix is singular at working precision, regularizing...
  design_matrix = make_first_level_design_matrix(


  Design matrix shape: (1005, 59)
  FD scrubbing: 12/1005 frames (1.2%)
  Processing v1_all...


/Volumes/drive/AVP-BDD/codes/.venv/lib/python3.9/site-packages/joblib/memory.py:326: UserWarning: imgs are being resampled to the mask_img resolution. This process is memory intensive. You might want to provide a target_affine that is equal to the affine of the imgs or resample the mask beforehand to save memory and computation time.
  return self.func(*args, **kwargs)


[NiftiMasker.wrapped] Resampling images


...

[NiftiMasker.wrapped] Resampling images


/Volumes/drive/AVP-BDD/codes/.venv/lib/python3.9/site-packages/joblib/memory.py:326: UserWarning: imgs are being resampled to the mask_img resolution. This process is memory intensive. You might want to provide a target_affine that is equal to the affine of the imgs or resample the mask beforehand to save memory and computation time.
  return self.func(*args, **kwargs)


[NiftiMasker.wrapped] Resampling images
    Time series shape: (1005, 2713) (2713 voxels)

GLM #2: Categorical ROIs (MNI space)
  Loaded 3 fMRI images in MNI space


/var/folders/hq/ms5zk3kj1pd2kwh1bnvtcr0m0000gn/T/ipykernel_12338/1092442517.py:219: UserWarning: Matrix is singular at working precision, regularizing...
  design_matrix = make_first_level_design_matrix(
/var/folders/hq/ms5zk3kj1pd2kwh1bnvtcr0m0000gn/T/ipykernel_12338/1092442517.py:219: UserWarning: Matrix is singular at working precision, regularizing...
  design_matrix = make_first_level_design_matrix(
/var/folders/hq/ms5zk3kj1pd2kwh1bnvtcr0m0000gn/T/ipykernel_12338/1092442517.py:219: UserWarning: Matrix is singular at working precision, regularizing...
  design_matrix = make_first_level_design_matrix(


  Design matrix shape: (1005, 48)
  Processing face_roi...
    Time series shape: (1005, 152) (152 voxels)
  Processing body_roi...
    Time series shape: (1005, 276) (276 voxels)
  Processing house_roi...
    Time series shape: (1005, 169) (169 voxels)

  Creating visualization...
  Saved: /Volumes/drive/AVP-BDD/results/high-level/glm_results/individual_subjects/sub-230_glm_summary.png

CREATING GROUP-LEVEL VISUALIZATIONS

  Creating visualizations for v1_all...

  Creating visualizations for face_roi...

  Creating visualizations for body_roi...

  Creating visualizations for house_roi...
  Group visualizations complete

CREATING HTML QC REPORT
  HTML report saved: /Volumes/drive/AVP-BDD/results/high-level/glm_results/QC_report_high_level_glm_20260421_181123.html

SAVING RESULTS

✅ All GLM results saved: /Volumes/drive/AVP-BDD/results/high-level/glm_results/all_subject_glm_results_20260422_114157.pkl
✅ Summary CSVs saved

ANALYSIS COMPLETE

Total subjects analyzed: 56

All results sa

## Second-level 

In [21]:
import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.formula.api import mixedlm
from statsmodels.stats.multitest import multipletests
import os

# -----------------------------
# Configuration
# -----------------------------

# Load the GLM results
results_file = '/Volumes/drive/AVP-BDD/results/high-level/glm_results/all_subject_glm_results_20260422_114157.pkl'  # Update with your actual timestamp

# Define output directory
output_dir = '/Volumes/drive/AVP-BDD/results/high-level/group_analysis'
os.makedirs(output_dir, exist_ok=True)

# Define subject groups
patients = subjects[0:30]  # BDD
controls = subjects[30:]   # HC

print("="*80)
print("HIGH-LEVEL TASK: LINEAR MIXED MODEL ANALYSIS")
print("Model: beta ~ group * SF + (1|subject)")
print("="*80)

# -----------------------------
# Load GLM Results
# -----------------------------
print("\nLoading GLM results...")

with open(results_file, 'rb') as f:
    all_subject_results = pickle.load(f)

print(f"Loaded results for {len(all_subject_results)} subjects")

# -----------------------------
# Define Analysis Structure
# -----------------------------

# Categories and conditions
categories = ['Face', 'Body', 'House']
sf_conditions = ['LSF', 'NSF', 'HSF']

# Category-specific ROIs
category_roi_map = {
    'Face': 'face_roi',
    'Body': 'body_roi',
    'House': 'house_roi'
}

# -----------------------------
# Helper Functions
# -----------------------------

def extract_lmm_data(all_results, category, roi_name, patients, controls):
    """Extract data in long format for LMM analysis"""
    data = []
    
    for sub in all_results.keys():
        group = 'BDD' if sub in patients else 'HC'
        
        # Check if ROI exists
        if roi_name not in all_results[sub]:
            continue
        
        roi_data = all_results[sub][roi_name]
        
        # Extract beta values for each SF condition
        for sf in sf_conditions:
            condition_name = f'{category}_{sf}'
            
            if condition_name not in roi_data:
                continue
            
            beta_value = roi_data[condition_name]['mean_beta']
            
            data.append({
                'subject': sub,
                'group': group,
                'SF': sf,
                'beta': beta_value
            })
    
    return pd.DataFrame(data)

def run_lmm_analysis(df, analysis_name):
    """Run linear mixed model with group × SF interaction"""
    
    # Convert to categorical
    df['group'] = pd.Categorical(df['group'], categories=['HC', 'BDD'])
    df['SF'] = pd.Categorical(df['SF'], categories=['NSF', 'LSF', 'HSF'])  # NSF as reference
    
    # Descriptive statistics
    desc_stats = df.groupby(['group', 'SF'])['beta'].agg(['mean', 'std', 'sem', 'count'])
    
    # Fit LMM with group × SF interaction
    model_formula = "beta ~ C(group, Treatment('HC')) * C(SF, Treatment('NSF'))"
    model = mixedlm(model_formula, df, groups=df["subject"])
    result = model.fit(reml=True)
    
    # Extract key statistics
    params = result.params
    pvalues = result.pvalues
    conf_int = result.conf_int()
    
    # Get specific effects
    group_effect_key = "C(group, Treatment('HC'))[T.BDD]"
    lsf_effect_key = "C(SF, Treatment('NSF'))[T.LSF]"
    hsf_effect_key = "C(SF, Treatment('NSF'))[T.HSF]"
    group_lsf_interaction_key = "C(group, Treatment('HC'))[T.BDD]:C(SF, Treatment('NSF'))[T.LSF]"
    group_hsf_interaction_key = "C(group, Treatment('HC'))[T.BDD]:C(SF, Treatment('NSF'))[T.HSF]"
    
    group_p = pvalues.get(group_effect_key, np.nan)
    lsf_p = pvalues.get(lsf_effect_key, np.nan)
    hsf_p = pvalues.get(hsf_effect_key, np.nan)
    group_lsf_int_p = pvalues.get(group_lsf_interaction_key, np.nan)
    group_hsf_int_p = pvalues.get(group_hsf_interaction_key, np.nan)
    
    # Follow-up pairwise comparisons
    pairwise_results = []
    
    # Within-group SF comparisons
    for group in ['BDD', 'HC']:
        # LSF vs NSF
        group_lsf = df[(df['group'] == group) & (df['SF'] == 'LSF')]['beta']
        group_nsf = df[(df['group'] == group) & (df['SF'] == 'NSF')]['beta']
        if len(group_lsf) > 0 and len(group_nsf) > 0:
            t_stat, p_val = stats.ttest_rel(group_lsf, group_nsf)
            mean_diff = group_lsf.mean() - group_nsf.mean()
            std_diff = (group_lsf - group_nsf).std()
            cohens_d = mean_diff / std_diff if std_diff > 0 else np.nan
            
            pairwise_results.append({
                'comparison': f'{group}: LSF vs NSF',
                'mean_diff': mean_diff,
                't_statistic': t_stat,
                'p_value': p_val,
                'cohens_d': cohens_d,
                'n': len(group_lsf)
            })
        
        # HSF vs NSF
        group_hsf = df[(df['group'] == group) & (df['SF'] == 'HSF')]['beta']
        if len(group_hsf) > 0 and len(group_nsf) > 0:
            t_stat, p_val = stats.ttest_rel(group_hsf, group_nsf)
            mean_diff = group_hsf.mean() - group_nsf.mean()
            std_diff = (group_hsf - group_nsf).std()
            cohens_d = mean_diff / std_diff if std_diff > 0 else np.nan
            
            pairwise_results.append({
                'comparison': f'{group}: HSF vs NSF',
                'mean_diff': mean_diff,
                't_statistic': t_stat,
                'p_value': p_val,
                'cohens_d': cohens_d,
                'n': len(group_hsf)
            })
        
        # LSF vs HSF
        if len(group_lsf) > 0 and len(group_hsf) > 0:
            t_stat, p_val = stats.ttest_rel(group_lsf, group_hsf)
            mean_diff = group_lsf.mean() - group_hsf.mean()
            std_diff = (group_lsf - group_hsf).std()
            cohens_d = mean_diff / std_diff if std_diff > 0 else np.nan
            
            pairwise_results.append({
                'comparison': f'{group}: LSF vs HSF',
                'mean_diff': mean_diff,
                't_statistic': t_stat,
                'p_value': p_val,
                'cohens_d': cohens_d,
                'n': len(group_lsf)
            })
    
    # Between-group comparisons for each SF
    for sf in ['LSF', 'NSF', 'HSF']:
        bdd_sf = df[(df['group'] == 'BDD') & (df['SF'] == sf)]['beta']
        hc_sf = df[(df['group'] == 'HC') & (df['SF'] == sf)]['beta']
        
        if len(bdd_sf) > 0 and len(hc_sf) > 0:
            t_stat, p_val = stats.ttest_ind(bdd_sf, hc_sf)
            pooled_std = np.sqrt((bdd_sf.std()**2 + hc_sf.std()**2) / 2)
            cohens_d = (hc_sf.mean() - bdd_sf.mean()) / pooled_std if pooled_std > 0 else np.nan
            
            pairwise_results.append({
                'comparison': f'{sf}: BDD vs HC',
                'mean_diff': hc_sf.mean() - bdd_sf.mean(),
                't_statistic': t_stat,
                'p_value': p_val,
                'cohens_d': cohens_d,
                'n': len(bdd_sf) + len(hc_sf)
            })
    
    # FDR correction on pairwise comparisons
    pairwise_df = pd.DataFrame(pairwise_results)
    if len(pairwise_df) > 0:
        _, pvals_corrected, _, _ = multipletests(pairwise_df['p_value'], alpha=0.05, method='fdr_bh')
        pairwise_df['p_fdr'] = pvals_corrected
        pairwise_df['sig_fdr'] = pvals_corrected < 0.05
    
    # One-sample t-tests vs zero
    one_sample_results = []
    for group in ['BDD', 'HC']:
        for sf in ['LSF', 'NSF', 'HSF']:
            data = df[(df['group'] == group) & (df['SF'] == sf)]['beta']
            if len(data) > 0:
                t_stat, p_val = stats.ttest_1samp(data, 0)
                one_sample_results.append({
                    'condition': f'{group} - {sf}',
                    'mean': data.mean(),
                    'std': data.std(),
                    't_statistic': t_stat,
                    'p_value': p_val,
                    'n': len(data)
                })
    
    one_sample_df = pd.DataFrame(one_sample_results)
    if len(one_sample_df) > 0:
        _, pvals_corrected_one, _, _ = multipletests(one_sample_df['p_value'], alpha=0.05, method='fdr_bh')
        one_sample_df['p_fdr'] = pvals_corrected_one
        one_sample_df['sig_fdr'] = pvals_corrected_one < 0.05
    
    return {
        'analysis': analysis_name,
        'model_result': result,
        'descriptive_stats': desc_stats,
        'group_p': group_p,
        'lsf_p': lsf_p,
        'hsf_p': hsf_p,
        'group_lsf_int_p': group_lsf_int_p,
        'group_hsf_int_p': group_hsf_int_p,
        'pairwise_df': pairwise_df,
        'one_sample_df': one_sample_df,
        'data': df
    }

def create_lmm_plots(result_dict, output_prefix):
    """Create visualization plots for LMM results"""
    
    df = result_dict['data']
    analysis_name = result_dict['analysis']
    
    colors = {'BDD': '#f5576c', 'HC': '#4facfe'}
    
    # Figure 1: Interaction plot + Bar plot + Violin plot
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle(f'{analysis_name}', fontsize=16, fontweight='bold')
    
    # Plot 1: Interaction plot
    ax1 = axes[0]
    sf_data = df.groupby(['group', 'SF'])['beta'].agg(['mean', 'sem']).reset_index()
    for group in ['BDD', 'HC']:
        group_data = sf_data[sf_data['group'] == group]
        # Order: NSF, LSF, HSF
        sf_order = ['NSF', 'LSF', 'HSF']
        group_data_ordered = group_data.set_index('SF').loc[sf_order].reset_index()
        ax1.errorbar([0, 1, 2], group_data_ordered['mean'], yerr=group_data_ordered['sem'],
                     marker='o', markersize=12, linewidth=2.5, capsize=8,
                     label=group, color=colors[group])
    
    ax1.set_xticks([0, 1, 2])
    ax1.set_xticklabels(['NSF', 'LSF', 'HSF'], fontsize=11)
    ax1.set_xlabel('Spatial Frequency', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Beta Value', fontsize=12, fontweight='bold')
    ax1.set_title('Group × SF Interaction', fontsize=12, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    ax1.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
    
    # Add interaction p-values if significant
    int_text = []
    if result_dict['group_lsf_int_p'] < 0.05:
        int_text.append(f"Group×LSF: p={result_dict['group_lsf_int_p']:.4f}*")
    if result_dict['group_hsf_int_p'] < 0.05:
        int_text.append(f"Group×HSF: p={result_dict['group_hsf_int_p']:.4f}*")
    
    if int_text:
        ax1.text(0.5, 0.95, '\n'.join(int_text),
                transform=ax1.transAxes, ha='center', va='top', fontsize=9,
                bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5))
    
    # Plot 2: Bar plot
    ax2 = axes[1]
    x = np.arange(3)
    width = 0.35
    
    bdd_nsf = df[(df['group'] == 'BDD') & (df['SF'] == 'NSF')]['beta']
    bdd_lsf = df[(df['group'] == 'BDD') & (df['SF'] == 'LSF')]['beta']
    bdd_hsf = df[(df['group'] == 'BDD') & (df['SF'] == 'HSF')]['beta']
    hc_nsf = df[(df['group'] == 'HC') & (df['SF'] == 'NSF')]['beta']
    hc_lsf = df[(df['group'] == 'HC') & (df['SF'] == 'LSF')]['beta']
    hc_hsf = df[(df['group'] == 'HC') & (df['SF'] == 'HSF')]['beta']
    
    bdd_means = [bdd_nsf.mean(), bdd_lsf.mean(), bdd_hsf.mean()]
    bdd_sems = [bdd_nsf.sem(), bdd_lsf.sem(), bdd_hsf.sem()]
    hc_means = [hc_nsf.mean(), hc_lsf.mean(), hc_hsf.mean()]
    hc_sems = [hc_nsf.sem(), hc_lsf.sem(), hc_hsf.sem()]
    
    ax2.bar(x - width/2, bdd_means, width, yerr=bdd_sems, label='BDD',
            capsize=5, alpha=0.8, color=colors['BDD'], edgecolor='black', linewidth=1.5)
    ax2.bar(x + width/2, hc_means, width, yerr=hc_sems, label='HC',
            capsize=5, alpha=0.8, color=colors['HC'], edgecolor='black', linewidth=1.5)
    
    ax2.set_ylabel('Beta Value', fontsize=12, fontweight='bold')
    ax2.set_title('Mean Effects by Group', fontsize=12, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(['NSF', 'LSF', 'HSF'], fontsize=11)
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3, axis='y')
    ax2.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
    
    # Plot 3: Violin plot
    ax3 = axes[2]
    # Reorder for plotting
    df_plot = df.copy()
    df_plot['SF'] = pd.Categorical(df_plot['SF'], categories=['NSF', 'LSF', 'HSF'], ordered=True)
    sns.violinplot(data=df_plot, x='SF', y='beta', hue='group', split=False, ax=ax3,
                   palette=colors, alpha=0.7)
    ax3.set_xlabel('Spatial Frequency', fontsize=12, fontweight='bold')
    ax3.set_ylabel('Beta Value', fontsize=12, fontweight='bold')
    ax3.set_title('Distribution of Effects', fontsize=12, fontweight='bold')
    ax3.legend(title='Group', fontsize=10)
    ax3.grid(True, alpha=0.3, axis='y')
    ax3.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.5)
    
    plt.tight_layout()
    plt.savefig(f'{output_prefix}_lmm_visualization.png', dpi=300, bbox_inches='tight')
    plt.close()
    
    # Figure 2: Individual subjects plot
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle(f'{analysis_name} - Individual Subjects', fontsize=16, fontweight='bold')
    
    for idx, sf in enumerate(['NSF', 'LSF', 'HSF']):
        ax = axes[idx]
        
        sf_data = df[df['SF'] == sf].copy()
        sf_data = sf_data.sort_values(['group', 'beta'])
        
        bdd_data = sf_data[sf_data['group'] == 'BDD']
        hc_data = sf_data[sf_data['group'] == 'HC']
        
        x_bdd = np.arange(len(bdd_data))
        x_hc = np.arange(len(hc_data)) + len(bdd_data) + 1
        
        ax.bar(x_bdd, bdd_data['beta'], color=colors['BDD'], alpha=0.7,
               edgecolor='black', linewidth=1)
        ax.bar(x_hc, hc_data['beta'], color=colors['HC'], alpha=0.7,
               edgecolor='black', linewidth=1)
        
        ax.axhline(bdd_data['beta'].mean(), color=colors['BDD'],
                   linestyle='--', linewidth=2,
                   label=f'BDD mean = {bdd_data["beta"].mean():.4f}')
        ax.axhline(hc_data['beta'].mean(), color=colors['HC'],
                   linestyle='--', linewidth=2,
                   label=f'HC mean = {hc_data["beta"].mean():.4f}')
        ax.axhline(0, color='black', linestyle='-', linewidth=1, alpha=0.3)
        
        ax.set_xlabel('Subject (sorted by group)', fontsize=12, fontweight='bold')
        ax.set_ylabel('Beta Value', fontsize=12, fontweight='bold')
        ax.set_title(f'{sf}', fontsize=12, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3, axis='y')
        
        ax.text(len(bdd_data)/2, ax.get_ylim()[0], 'BDD', ha='center', va='top',
                fontsize=12, fontweight='bold', color=colors['BDD'])
        ax.text(len(bdd_data) + 1 + len(hc_data)/2, ax.get_ylim()[0], 'HC', ha='center', va='top',
                fontsize=12, fontweight='bold', color=colors['HC'])
    
    plt.tight_layout()
    plt.savefig(f'{output_prefix}_individual_subjects.png', dpi=300, bbox_inches='tight')
    plt.close()

# -----------------------------
# Run LMM Analyses
# -----------------------------
print("\n" + "="*80)
print("RUNNING LMM ANALYSES")
print("="*80)

all_lmm_results = {}

for category in categories:
    # Analysis 1: Category in its specific ROI
    roi = category_roi_map[category]
    print(f"\n{'-'*80}")
    print(f"Analyzing {category} stimuli in {roi}")
    print(f"{'-'*80}")
    
    df = extract_lmm_data(all_subject_results, category, roi, patients, controls)
    
    if len(df) == 0:
        print(f"  Warning: No data found, skipping")
        continue
    
    print(f"  Data shape: {df.shape}")
    print(f"  Subjects: {len(df['subject'].unique())}")
    print(f"  Observations per subject: {df.groupby('subject').size().value_counts().to_dict()}")
    
    # Run LMM
    result_dict = run_lmm_analysis(df, f'{category} in {roi.replace("_", " ").title()}')
    all_lmm_results[f'{category}_{roi}'] = result_dict
    
    # Print results
    print("\nDescriptive Statistics:")
    print(result_dict['descriptive_stats'])
    
    print("\nLMM Fixed Effects:")
    print(f"  Main effect of group: p = {result_dict['group_p']:.4f}")
    print(f"  Main effect of LSF (vs NSF): p = {result_dict['lsf_p']:.4f}")
    print(f"  Main effect of HSF (vs NSF): p = {result_dict['hsf_p']:.4f}")
    print(f"  Group × LSF interaction: p = {result_dict['group_lsf_int_p']:.4f}")
    print(f"  Group × HSF interaction: p = {result_dict['group_hsf_int_p']:.4f}")
    
    print("\nPairwise Comparisons (FDR corrected):")
    print(result_dict['pairwise_df'].to_string(index=False))
    
    sig_pairwise = result_dict['pairwise_df'][result_dict['pairwise_df']['sig_fdr']]
    if len(sig_pairwise) > 0:
        print("\nSignificant pairwise comparisons:")
        for _, row in sig_pairwise.iterrows():
            print(f"  • {row['comparison']}: d = {row['cohens_d']:.3f}, p_fdr = {row['p_fdr']:.4f}")
    
    # Save results
    with open(os.path.join(output_dir, f'{category}_{roi}_lmm_full_results.txt'), 'w') as f:
        f.write(str(result_dict['model_result'].summary()))
    
    result_dict['pairwise_df'].to_csv(
        os.path.join(output_dir, f'{category}_{roi}_pairwise_comparisons.csv'),
        index=False
    )
    result_dict['one_sample_df'].to_csv(
        os.path.join(output_dir, f'{category}_{roi}_one_sample_tests.csv'),
        index=False
    )
    
    # Create plots
    create_lmm_plots(result_dict, os.path.join(output_dir, f'{category}_{roi}'))
    print(f"  ✓ Saved visualizations")
    
    # Analysis 2: Category in v1_all
    print(f"\n{'-'*80}")
    print(f"Analyzing {category} stimuli in v1_all")
    print(f"{'-'*80}")
    
    df = extract_lmm_data(all_subject_results, category, 'v1_all', patients, controls)
    
    if len(df) == 0:
        print(f"  Warning: No data found, skipping")
        continue
    
    print(f"  Data shape: {df.shape}")
    print(f"  Subjects: {len(df['subject'].unique())}")
    print(f"  Observations per subject: {df.groupby('subject').size().value_counts().to_dict()}")
    
    # Run LMM
    result_dict = run_lmm_analysis(df, f'{category} in V1 All')
    all_lmm_results[f'{category}_v1_all'] = result_dict
    
    # Print results
    print("\nDescriptive Statistics:")
    print(result_dict['descriptive_stats'])
    
    print("\nLMM Fixed Effects:")
    print(f"  Main effect of group: p = {result_dict['group_p']:.4f}")
    print(f"  Main effect of LSF (vs NSF): p = {result_dict['lsf_p']:.4f}")
    print(f"  Main effect of HSF (vs NSF): p = {result_dict['hsf_p']:.4f}")
    print(f"  Group × LSF interaction: p = {result_dict['group_lsf_int_p']:.4f}")
    print(f"  Group × HSF interaction: p = {result_dict['group_hsf_int_p']:.4f}")
    
    print("\nPairwise Comparisons (FDR corrected):")
    print(result_dict['pairwise_df'].to_string(index=False))
    
    sig_pairwise = result_dict['pairwise_df'][result_dict['pairwise_df']['sig_fdr']]
    if len(sig_pairwise) > 0:
        print("\nSignificant pairwise comparisons:")
        for _, row in sig_pairwise.iterrows():
            print(f"  • {row['comparison']}: d = {row['cohens_d']:.3f}, p_fdr = {row['p_fdr']:.4f}")
    
    # Save results
    with open(os.path.join(output_dir, f'{category}_v1_all_lmm_full_results.txt'), 'w') as f:
        f.write(str(result_dict['model_result'].summary()))
    
    result_dict['pairwise_df'].to_csv(
        os.path.join(output_dir, f'{category}_v1_all_pairwise_comparisons.csv'),
        index=False
    )
    result_dict['one_sample_df'].to_csv(
        os.path.join(output_dir, f'{category}_v1_all_one_sample_tests.csv'),
        index=False
    )
    
    # Create plots
    create_lmm_plots(result_dict, os.path.join(output_dir, f'{category}_v1_all'))
    print(f"  ✓ Saved visualizations")

# -----------------------------
# Summary Report
# -----------------------------
print("\n" + "="*80)
print("GENERATING SUMMARY REPORT")
print("="*80)

summary_report = os.path.join(output_dir, 'high_level_lmm_summary.txt')

with open(summary_report, 'w') as f:
    f.write("="*80 + "\n")
    f.write("HIGH-LEVEL TASK: LINEAR MIXED MODEL ANALYSIS SUMMARY\n")
    f.write("Model: beta ~ group * SF + (1|subject)\n")
    f.write("="*80 + "\n\n")
    
    for key, result_dict in all_lmm_results.items():
        f.write(f"{result_dict['analysis']}:\n\n")
        f.write("Fixed Effects:\n")
        f.write(f"  Main effect of group: p = {result_dict['group_p']:.4f}\n")
        f.write(f"  Main effect of LSF (vs NSF): p = {result_dict['lsf_p']:.4f}\n")
        f.write(f"  Main effect of HSF (vs NSF): p = {result_dict['hsf_p']:.4f}\n")
        f.write(f"  Group × LSF interaction: p = {result_dict['group_lsf_int_p']:.4f}\n")
        f.write(f"  Group × HSF interaction: p = {result_dict['group_hsf_int_p']:.4f}\n\n")
        
        f.write("Pairwise Comparisons (FDR corrected):\n")
        for _, row in result_dict['pairwise_df'].iterrows():
            sig = '***' if row['p_fdr'] < 0.001 else '**' if row['p_fdr'] < 0.01 else '*' if row['p_fdr'] < 0.05 else ''
            f.write(f"  {row['comparison']}: d = {row['cohens_d']:.3f}, p_fdr = {row['p_fdr']:.4f} {sig}\n")
        f.write("\n")
        
        sig_pairwise = result_dict['pairwise_df'][result_dict['pairwise_df']['sig_fdr']]
        if len(sig_pairwise) > 0:
            f.write("Significant comparisons:\n")
            for _, row in sig_pairwise.iterrows():
                f.write(f"  • {row['comparison']}\n")
        f.write("\n" + "-"*80 + "\n\n")

print(f"✓ Summary report saved: {summary_report}")

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)
print(f"\nAll results saved to: {output_dir}")

# Print summary of significant findings
print("\nSummary of Significant Effects:")
all_sig = []
for key, result_dict in all_lmm_results.items():
    if result_dict['group_lsf_int_p'] < 0.05:
        all_sig.append(f"{result_dict['analysis']}: Significant Group × LSF interaction (p={result_dict['group_lsf_int_p']:.4f})")
    if result_dict['group_hsf_int_p'] < 0.05:
        all_sig.append(f"{result_dict['analysis']}: Significant Group × HSF interaction (p={result_dict['group_hsf_int_p']:.4f})")
    if result_dict['group_p'] < 0.05:
        all_sig.append(f"{result_dict['analysis']}: Significant main effect of group (p={result_dict['group_p']:.4f})")
    if result_dict['lsf_p'] < 0.05:
        all_sig.append(f"{result_dict['analysis']}: Significant main effect of LSF (p={result_dict['lsf_p']:.4f})")
    if result_dict['hsf_p'] < 0.05:
        all_sig.append(f"{result_dict['analysis']}: Significant main effect of HSF (p={result_dict['hsf_p']:.4f})")

if len(all_sig) > 0:
    for finding in all_sig:
        print(f"  • {finding}")
else:
    print("  No significant main effects or interactions found")

print(f"\nTotal analyses run: {len(all_lmm_results)}")
print(f"Files generated per analysis:")
print(f"  - LMM full results (txt)")
print(f"  - Pairwise comparisons (csv)")
print(f"  - One-sample tests (csv)")
print(f"  - 2 visualization figures (png)")

HIGH-LEVEL TASK: LINEAR MIXED MODEL ANALYSIS
Model: beta ~ group * SF + (1|subject)

Loading GLM results...
Loaded results for 56 subjects

RUNNING LMM ANALYSES

--------------------------------------------------------------------------------
Analyzing Face stimuli in face_roi
--------------------------------------------------------------------------------
  Data shape: (156, 4)
  Subjects: 52
  Observations per subject: {3: 52}


/var/folders/hq/ms5zk3kj1pd2kwh1bnvtcr0m0000gn/T/ipykernel_12338/3398170778.py:99: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  desc_stats = df.groupby(['group', 'SF'])['beta'].agg(['mean', 'std', 'sem', 'count'])



Descriptive Statistics:
                mean        std       sem  count
group SF                                        
HC    NSF  13.690342  24.509299  4.901860     25
      LSF  13.221821  18.363051  3.672610     25
      HSF   7.346805  18.005353  3.601071     25
BDD   NSF   8.016656  16.716033  3.217002     27
      LSF  10.939975  23.399988  4.503330     27
      HSF   8.645128  19.310726  3.716351     27

LMM Fixed Effects:
  Main effect of group: p = 0.3127
  Main effect of LSF (vs NSF): p = 0.9300
  Main effect of HSF (vs NSF): p = 0.2345
  Group × LSF interaction: p = 0.6469
  Group × HSF interaction: p = 0.3464

Pairwise Comparisons (FDR corrected):
     comparison  mean_diff  t_statistic  p_value  cohens_d  n    p_fdr  sig_fdr
BDD: LSF vs NSF   2.923319     0.534439 0.597579       NaN 27 0.921927    False
BDD: HSF vs NSF   0.628472     0.121392 0.904314       NaN 27 0.921927    False
BDD: LSF vs HSF   2.294847     0.386321 0.702403       NaN 27 0.921927    False
 HC: LSF 

/var/folders/hq/ms5zk3kj1pd2kwh1bnvtcr0m0000gn/T/ipykernel_12338/3398170778.py:256: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sf_data = df.groupby(['group', 'SF'])['beta'].agg(['mean', 'sem']).reset_index()


  ✓ Saved visualizations

--------------------------------------------------------------------------------
Analyzing Face stimuli in v1_all
--------------------------------------------------------------------------------
  Data shape: (150, 4)
  Subjects: 50
  Observations per subject: {3: 50}

Descriptive Statistics:
                mean        std       sem  count
group SF                                        
HC    NSF  20.968069  30.348470  6.194855     24
      LSF  19.817612  26.078224  5.323195     24
      HSF  14.981050  28.158730  5.747877     24
BDD   NSF  10.899871  37.742511  7.401915     26
      LSF  22.723358  40.181807  7.880301     26
      HSF  16.266359  36.509787  7.160158     26

LMM Fixed Effects:
  Main effect of group: p = 0.2924
  Main effect of LSF (vs NSF): p = 0.9013
  Main effect of HSF (vs NSF): p = 0.5187
  Group × LSF interaction: p = 0.3133
  Group × HSF interaction: p = 0.3776

Pairwise Comparisons (FDR corrected):
     comparison  mean_diff  t_stat

...

  ✓ Saved visualizations

--------------------------------------------------------------------------------
Analyzing House stimuli in house_roi
--------------------------------------------------------------------------------
  Data shape: (156, 4)
  Subjects: 52
  Observations per subject: {3: 52}

Descriptive Statistics:
                mean        std       sem  count
group SF                                        
HC    NSF   3.700040  16.282159  3.256432     25
      LSF   8.167462  13.336843  2.667369     25
      HSF  10.721444  13.771534  2.754307     25
BDD   NSF  11.638848  19.177323  3.690678     27
      LSF   9.973178  13.965623  2.687685     27
      HSF   7.227092  13.376091  2.574230     27

LMM Fixed Effects:
  Main effect of group: p = 0.0592
  Main effect of LSF (vs NSF): p = 0.2365
  Main effect of HSF (vs NSF): p = 0.0628
  Group × LSF interaction: p = 0.2416
  Group × HSF interaction: p = 0.0290

Pairwise Comparisons (FDR corrected):
     comparison  mean_diff  t_

/var/folders/hq/ms5zk3kj1pd2kwh1bnvtcr0m0000gn/T/ipykernel_12338/3398170778.py:99: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  desc_stats = df.groupby(['group', 'SF'])['beta'].agg(['mean', 'std', 'sem', 'count'])
/var/folders/hq/ms5zk3kj1pd2kwh1bnvtcr0m0000gn/T/ipykernel_12338/3398170778.py:256: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sf_data = df.groupby(['group', 'SF'])['beta'].agg(['mean', 'sem']).reset_index()


  ✓ Saved visualizations

--------------------------------------------------------------------------------
Analyzing House stimuli in v1_all
--------------------------------------------------------------------------------
  Data shape: (150, 4)
  Subjects: 50
  Observations per subject: {3: 50}

Descriptive Statistics:
                mean        std       sem  count
group SF                                        
HC    NSF  17.751139  29.490146  6.019651     24
      LSF  29.803613  29.298931  5.980619     24
      HSF  27.305154  22.355097  4.563215     24
BDD   NSF  10.826580  33.996569  6.667276     26
      LSF  28.443183  37.099980  7.275905     26
      HSF  10.054798  28.201835  5.530835     26

LMM Fixed Effects:
  Main effect of group: p = 0.4233
  Main effect of LSF (vs NSF): p = 0.1434
  Main effect of HSF (vs NSF): p = 0.2460
  Group × LSF interaction: p = 0.6261
  Group × HSF interaction: p = 0.3660

Pairwise Comparisons (FDR corrected):
     comparison  mean_diff  t_sta

/var/folders/hq/ms5zk3kj1pd2kwh1bnvtcr0m0000gn/T/ipykernel_12338/3398170778.py:99: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  desc_stats = df.groupby(['group', 'SF'])['beta'].agg(['mean', 'std', 'sem', 'count'])
/var/folders/hq/ms5zk3kj1pd2kwh1bnvtcr0m0000gn/T/ipykernel_12338/3398170778.py:256: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sf_data = df.groupby(['group', 'SF'])['beta'].agg(['mean', 'sem']).reset_index()


  ✓ Saved visualizations

GENERATING SUMMARY REPORT
✓ Summary report saved: /Volumes/drive/AVP-BDD/results/high-level/group_analysis/high_level_lmm_summary.txt

ANALYSIS COMPLETE

All results saved to: /Volumes/drive/AVP-BDD/results/high-level/group_analysis

Summary of Significant Effects:
  • House in House Roi: Significant Group × HSF interaction (p=0.0290)

Total analyses run: 6
Files generated per analysis:
  - LMM full results (txt)
  - Pairwise comparisons (csv)
  - One-sample tests (csv)
  - 2 visualization figures (png)
